In [19]:
import os
import json
import re
import asyncio
import pandas as pd
from datetime import datetime, timezone, timedelta
from playwright.async_api import async_playwright

# =========================================================================
# 1. MODELO DE DATOS: FILTRADO POR MARZO
# =========================================================================
class TripActivity:
    def __init__(self, raw_data: dict):
        self.raw = raw_data
        self.data = {}
        self._parse()

    @staticmethod
    def _parse_money(text: str) -> float:
        if not text: return 0.0
        cleaned = re.sub(r'[^\d\.,]', '', text)
        if ',' in cleaned and '.' in cleaned:
            cleaned = cleaned.replace('.', '').replace(',', '.') if cleaned.rfind(',') > cleaned.rfind('.') else cleaned.replace(',', '')
        elif ',' in cleaned:
            cleaned = cleaned.replace(',', '.') if len(cleaned) - cleaned.rfind(',') == 3 else cleaned.replace(',', '')
        try: return float(cleaned)
        except: return 0.0

    @staticmethod
    def _parse_timestamp(ts) -> datetime:
        try:
            return datetime.fromtimestamp(int(ts), tz=timezone.utc)
        except:
            return None

    def _parse(self):
        ts_utc = self._parse_timestamp(self.raw.get('recognizedAt', 0))
        
        # AJUSTE DEFINITIVO PARA MORELIA (UTC-6) + CORTE UBER (4:00 AM)
        # Restamos 10 horas al timestamp UTC para que:
        # Lunes 04:00 AM local sea el inicio exacto del lunes (00:00) en el script.
        ts_uber = ts_utc - timedelta(hours=10) if ts_utc else None
        
        # Obtenemos la semana ISO para filtrar igual que los extractos
        semana_iso = ts_uber.isocalendar().week if ts_uber else 0
        
        self.data = {
            'trip_id': self.raw.get('uuid', ''),
            'year': ts_uber.year if ts_uber else 0,
            'semana': semana_iso,
            'status': self.raw.get('status', ''),
            'fare_amount': self._parse_money(self.raw.get('formattedTotal', '')),
        }

    @property
    def is_valid_march(self) -> bool:
        # RANGO DE MARZO SEGÚN TUS IMÁGENES:
        # Semana 9 (Inicia 24/Feb) hasta Semana 14 (Termina 07/Abr)
        semanas_marzo = [9, 10, 11, 12, 13, 14]
        
        return (self.data['year'] == 2025 and 
                self.data['semana'] in semanas_marzo and 
                self.data['fare_amount'] > 0)

# =========================================================================
# 2. MANAGER: REPORTE ESPECÍFICO DE MARZO
# =========================================================================
class UberTripManager:
    def __init__(self, raw_json_list: list):
        # Filtramos solo viajes de MARZO usando la nueva propiedad
        self.valid_trips = [TripActivity(t).data for t in raw_json_list if TripActivity(t).is_valid_march]
        self.df = pd.DataFrame(self.valid_trips)
        if not self.df.empty:
            self.df = self.df.drop_duplicates(subset=['trip_id'])

    def generar_reporte_marzo(self):
        if self.df.empty:
            print("❌ No se encontraron viajes completados en el mes de MARZO.")
            return

        # Agrupamos por año para distinguir Marzo 2024 de Marzo 2025
        reporte = self.df.groupby('year').agg(
            total_viajes=('trip_id', 'count'),
            ingreso_bruto_mxn=('fare_amount', 'sum'),
            ticket_promedio=('fare_amount', 'mean')
        ).round(2)

        print("\n🌸 --- REPORTE DE MARZO (UBER) ---")
        print(reporte)
        
        # Guardamos un CSV específico de Marzo
        filename = "viajes_solo_marzo.csv"
        self.df.to_csv(filename, index=False)
        print(f"✅ Detalle guardado en '{filename}'")

# =========================================================================
# 3. SCRAPER ASÍNCRONO (Mantiene la lógica de intercepción)
# =========================================================================
class AsyncUberDataScraper:
    def __init__(self, session_dir="uber_session", session_file="gs_session.json"):
        self.session_dir = session_dir
        self.session_file = os.path.join(self.session_dir, session_file)
        self.json_out = "viajes_marzo_capturados.json"
        self.viajes_acumulados = {}

    def _limpiar_archivos_viejos(self):
        if os.path.exists(self.json_out):
            os.remove(self.json_out)

    async def _interceptar_trafico(self, response):
        if "getWebActivityFeed" in response.url and response.status == 200:
            try:
                datos_json = await response.json()
                nuevos_viajes = datos_json.get('data', {}).get('activities', [])
                
                viajes_nuevos_count = 0
                for viaje in nuevos_viajes:
                    uuid = viaje.get('uuid')
                    if uuid and uuid not in self.viajes_acumulados:
                        self.viajes_acumulados[uuid] = viaje
                        viajes_nuevos_count += 1
                
                if viajes_nuevos_count > 0:
                    print(f"✅ {viajes_nuevos_count} viajes nuevos en memoria. (Total: {len(self.viajes_acumulados)})")
            except Exception:
                pass

    def _guardar_y_reportar(self):
        if not self.viajes_acumulados:
            print("⚠️ No se capturaron datos.")
            return

        lista_viajes = list(self.viajes_acumulados.values())
        
        # Guardamos TODO lo capturado en el JSON por si acaso
        with open(self.json_out, "w", encoding="utf-8") as f:
            json.dump(lista_viajes, f, ensure_ascii=False, indent=4)
        
        # El Manager se encarga de filtrar solo Marzo para el reporte
        manager = UberTripManager(lista_viajes)
        manager.generar_reporte_marzo()

    async def ejecutar(self):
        print("🚀 Iniciando scraper para MARZO...")
        self._limpiar_archivos_viejos()
        if not os.path.exists(self.session_dir): os.makedirs(self.session_dir)

        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=False)
            context = await browser.new_context(storage_state=self.session_file if os.path.exists(self.session_file) else None)
            
            page = await context.new_page()
            page.on("response", self._interceptar_trafico)

            await page.goto("https://drivers.uber.com/earnings/activities", wait_until="networkidle")

            print("\n💡 ESTRATEGIA PARA MARZO:")
            print("1. Carga la página y asegúrate de estar en 2024/2025.")
            print("2. Haz scroll lento hacia abajo. Cuando veas que entras a MARZO, ve más despacio.")
            print("3. No dejes de hacer scroll hasta que veas FEBRERO (para asegurar el mes completo).")
            print("4. Presiona ENTER aquí abajo cuando el contador de viajes deje de subir.")

            await asyncio.to_thread(input, "\n⏳ PRESIONA 'ENTER' PARA PROCESAR MARZO...\n")

            await context.storage_state(path=self.session_file)
            self._guardar_y_reportar()
            await browser.close()

# Ejecución
bot = AsyncUberDataScraper()
await bot.ejecutar()

🚀 Iniciando scraper para MARZO...

💡 ESTRATEGIA PARA MARZO:
1. Carga la página y asegúrate de estar en 2024/2025.
2. Haz scroll lento hacia abajo. Cuando veas que entras a MARZO, ve más despacio.
3. No dejes de hacer scroll hasta que veas FEBRERO (para asegurar el mes completo).
4. Presiona ENTER aquí abajo cuando el contador de viajes deje de subir.
✅ 15 viajes nuevos en memoria. (Total: 15)
✅ 15 viajes nuevos en memoria. (Total: 30)
✅ 15 viajes nuevos en memoria. (Total: 45)
✅ 15 viajes nuevos en memoria. (Total: 60)
✅ 15 viajes nuevos en memoria. (Total: 75)
✅ 9 viajes nuevos en memoria. (Total: 84)
✅ 15 viajes nuevos en memoria. (Total: 99)
✅ 15 viajes nuevos en memoria. (Total: 114)
✅ 15 viajes nuevos en memoria. (Total: 129)
✅ 15 viajes nuevos en memoria. (Total: 144)
✅ 15 viajes nuevos en memoria. (Total: 159)
✅ 7 viajes nuevos en memoria. (Total: 166)
✅ 15 viajes nuevos en memoria. (Total: 181)
✅ 15 viajes nuevos en memoria. (Total: 196)
✅ 15 viajes nuevos en memoria. (Total: 2


⏳ PRESIONA 'ENTER' PARA PROCESAR MARZO...
 



🌸 --- REPORTE DE MARZO (UBER) ---
      total_viajes  ingreso_bruto_mxn  ticket_promedio
year                                                  
2025           497           25891.23             52.1
✅ Detalle guardado en 'viajes_solo_marzo.csv'


In [24]:
import pdfplumber
import re
import os
import glob
import pandas as pd

class UberReportExtractor:
    def __init__(self, file_path):
        self.file_path = file_path
        self.full_text = ""
        self.page1_text = ""
        self.texto_plano = "" 
        self.data = {
            "driver_name": "Roman Yair Ortega",
            "weekly_period": "Not found",
            "total_trips": 0,
            "connected_hours_decimal": 0.0,
            "gross_amount": 0.0,
            "tips_amount": 0.0,
            "incentives_amount": 0.0,
            "taxes_amount": 0.0
        }

    def load_pdf(self):
        try:
            with pdfplumber.open(self.file_path) as pdf:
                for index, page in enumerate(pdf.pages):
                    text = page.extract_text()
                    if text:
                        clean_text = text.replace("\u0000", "")
                        self.full_text += clean_text + "\n"
                        if index == 0:
                            self.page1_text = clean_text
            
            # Texto aplastado para forzar la adyacencia
            self.texto_plano = " ".join(self.full_text.split())
            return True
        except Exception as e:
            print(f"❌ Error loading PDF: {e}")
            return False

    def parse_data(self):
        # 1. Fechas 
        match_date = re.search(r"(\d{2}/\d{2}/\d{4}).*?(\d{2}/\d{2}/\d{4})", self.page1_text)
        if match_date: 
            self.data["weekly_period"] = f"Del {match_date.group(1)} al {match_date.group(2)}"
        else:
            match_date_alt = re.search(r"(\d{1,2}\s+[a-z]{3}\s+\d{4}.*?-.*?\d{4})", self.page1_text, re.IGNORECASE)
            if match_date_alt:
                self.data["weekly_period"] = re.sub(r"\s+\d{1,2}\s+[ap]\.\s+m\.", "", match_date_alt.group(1)).strip()

        # 2. Viajes
        match_trips = re.search(r"viajes completados[^\d]*(\d+)", self.full_text, re.IGNORECASE)
        if match_trips: self.data["total_trips"] = int(match_trips.group(1))
            
# 3. Tiempo (Cambiamos a .*? para que logre saltar la trampa del "P2+P3")
        match_time = re.search(r"Tiempo de trabajo efectivo.*?(\d+)\s*h.*?(\d+)\s*m", self.texto_plano, re.IGNORECASE)
        if match_time:
            self.data["connected_hours_decimal"] = round(int(match_time.group(1)) + (int(match_time.group(2)) / 60.0), 2)
        # =========================================================
        # 💰 CONVERSOR DE DINERO INTELIGENTE
        # =========================================================
        def to_float(val_str):
            val_str = re.sub(r'[^\d\.,]', '', val_str)
            if not val_str: return 0.0
            if ',' in val_str and '.' in val_str:
                if val_str.rfind(',') > val_str.rfind('.'): 
                    val_str = val_str.replace('.', '').replace(',', '.')
                else:
                    val_str = val_str.replace(',', '')
            elif ',' in val_str:
                if len(val_str) > 3 and val_str[-3] == ',':
                    val_str = val_str.replace(',', '.')
                else:
                    val_str = val_str.replace(',', '')
            return float(val_str)

        # 🌟 EL CANDADO DE ADYACENCIA ESTRICTA (\s+)
        # Obliga a que el número esté PEGADO a la frase, ignorando subtítulos engañosos.
        # Soporta los nombres nuevos y viejos de Uber.
        
        match_gross = re.search(r"(?:Importe bruto que has generado|Monto bruto generado)\s+\$?\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_gross: self.data["gross_amount"] = to_float(match_gross.group(1))

        match_tips = re.search(r"(?:Propinas dadas por los usuarios|Propinas otorgadas)\s+\$?\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_tips: self.data["tips_amount"] = to_float(match_tips.group(1))

# 6. Incentivos (Agregamos "Otras ganancias" al diccionario de Uber)
        match_incentives = re.search(r"(?:Otras ganancias|Ganancias adicionales|promoción y fomento)\s*\$?\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_incentives: self.data["incentives_amount"] = to_float(match_incentives.group(1))

        match_taxes = re.search(r"Impuestos\s*(?:-?\s*\$?|-)\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_taxes: self.data["taxes_amount"] = -abs(to_float(match_taxes.group(1)))

In [25]:
import os

# 1. Definir la carpeta objetivo
carpeta_marzo = "Uber_Reports/Marzo_2025"

# 2. Inicializar acumuladores para el Gran Total del mes
total_mes_viajes = 0
total_mes_dinero = 0.0
total_mes_horas = 0.0
archivos_procesados = 0

print(f"📂 Iniciando procesamiento de la carpeta: {carpeta_marzo}\n")

# 3. Bucle para procesar cada archivo
if os.path.exists(carpeta_marzo):
    # Ordenamos los archivos para que aparezcan en orden cronológico
    for archivo in sorted(os.listdir(carpeta_marzo)):
        if archivo.lower().endswith(".pdf"):
            ruta_completa = os.path.join(carpeta_marzo, archivo)
            
            extractor = UberReportExtractor(file_path=ruta_completa)
            
            if extractor.load_pdf():
                extractor.parse_data()
                d = extractor.data
                
                # Acumular para el Gran Total
                total_mes_viajes += d['total_trips']
                total_mes_dinero += d['gross_amount']
                total_mes_horas += d['connected_hours_decimal']
                f_tips  = f"${d['tips_amount']:,.2f}"
                archivos_procesados += 1
                
                # Formateo de datos individuales
                h = int(d['connected_hours_decimal'])
                m = int((d['connected_hours_decimal'] - h) * 60)
                f_time = f"{h}h {m}m"
                
                # IMPRESIÓN DEL TICKET SEMANAL
                print("-" * 50)
                print(f"📄 ARCHIVO: {archivo}")
                print("-" * 50)
                print(f" 📅 SEMANA:                {d['weekly_period']}")
                print(f" 🚗 VIAJES COMPLETADOS:    {d['total_trips']}")
                print(f" ⏱️ TIEMPO CONECTADO:      {f_time}")
                print(f" 🪙 PROPINAS:              {f_tips}")
                print(f" 💵 MONTO BRUTO:           ${d['gross_amount']:,.2f}")
                print(f" 💸 IMPUESTOS:             -${abs(d['taxes_amount']):,.2f}")
                print("-" * 50)
                print("\n")

    # 4. RESUMEN FINAL DEL MES
    print("=" * 50)
    print("🏆 RESUMEN TOTAL: MARZO 2025")
    print("=" * 50)
    print(f" 📂 Reportes procesados:    {archivos_procesados}")
    print(f" 🚗 Total Viajes:           {total_mes_viajes}")
    print(f" ⏱️ Total Horas:            {int(total_mes_horas)}h {int((total_mes_horas % 1)*60)}m")
    print(f" 💰 Ingreso Bruto Total:    ${total_mes_dinero:,.2f}")
    print(f" 📈 Promedio por Viaje:     ${(total_mes_dinero/total_mes_viajes if total_mes_viajes > 0 else 0):,.2f}")
    print("=" * 50)

else:
    print(f"❌ La carpeta '{carpeta_marzo}' no existe. Verifica la ruta.")

📂 Iniciando procesamiento de la carpeta: Uber_Reports/Marzo_2025

--------------------------------------------------
📄 ARCHIVO: resumen_10 de mar, 4_00.pdf
--------------------------------------------------
 📅 SEMANA:                Del 03/03/2025 al 10/03/2025
 🚗 VIAJES COMPLETADOS:    80
 ⏱️ TIEMPO CONECTADO:      26h 11m
 🪙 PROPINAS:              $15.00
 💵 MONTO BRUTO:           $4,567.21
 💸 IMPUESTOS:             -$260.62
--------------------------------------------------


--------------------------------------------------
📄 ARCHIVO: resumen_17 de mar, 4_00.pdf
--------------------------------------------------
 📅 SEMANA:                Del 10/03/2025 al 17/03/2025
 🚗 VIAJES COMPLETADOS:    105
 ⏱️ TIEMPO CONECTADO:      31h 1m
 🪙 PROPINAS:              $0.00
 💵 MONTO BRUTO:           $5,249.90
 💸 IMPUESTOS:             -$391.83
--------------------------------------------------


--------------------------------------------------
📄 ARCHIVO: resumen_24 de mar, 4_00.pdf
-----------

In [44]:
from datetime import datetime, timezone, timedelta
import re
import pandas as pd

class TripActivity:
    def __init__(self, raw_data: dict):
        self.raw = raw_data
        self.data = {}
        self._parse()

    @staticmethod
    def _parse_money(text: str) -> float:
        if not text: return 0.0
        cleaned = re.sub(r'[^\d\.,]', '', text)
        if ',' in cleaned and '.' in cleaned:
            cleaned = cleaned.replace('.', '').replace(',', '.') if cleaned.rfind(',') > cleaned.rfind('.') else cleaned.replace(',', '')
        elif ',' in cleaned:
            cleaned = cleaned.replace(',', '.') if len(cleaned) - cleaned.rfind(',') == 3 else cleaned.replace(',', '')
        try: return float(cleaned)
        except: return 0.0

    @staticmethod
    def _parse_duration(text: str) -> float:
        """Convierte '25 min 12 seg' a minutos totales (float)"""
        if not text: return 0.0
        m_match = re.search(r'(\d+)\s*min', text)
        s_match = re.search(r'(\d+)\s*seg', text)
        m = int(m_match.group(1)) if m_match else 0
        s = int(s_match.group(1)) if s_match else 0
        return round(m + (s / 60.0), 2)

    def _parse(self):
        # 1. Manejo seguro del timestamp
        ts_val = self.raw.get('recognizedAt')
        if ts_val is None:
            self.data = {'semana': 0, 'year': 0, 'monto': 0, 'duracion_min': 0, 'trip_id': ''}
            return

        ts_utc = datetime.fromtimestamp(int(ts_val), tz=timezone.utc)
        
        # AJUSTE MÉXICO (UTC-6) + CORTE UBER (4:00 AM) = -10 HORAS
        # Esto alinea tus datos con los extractos semanales oficiales
        ts_uber = ts_utc - timedelta(hours=10)
        
        # 2. Obtener metadatos de forma segura (meta or {})
        meta = self.raw.get('tripMetaData') or {}
        
        # 3. ASIGNACIÓN ÚNICA: Consolidamos financiero + geolocalización
        self.data = {
            'trip_id': self.raw.get('uuid', ''),
            'year': ts_uber.year,
            'semana': ts_uber.isocalendar().week,
            'fecha_local': ts_uber.strftime('%Y-%m-%d %H:%M:%S'),
            'monto': self._parse_money(self.raw.get('formattedTotal', '')),
            'duracion_min': self._parse_duration(meta.get('formattedDuration', '')),
            'distancia_km': self._parse_money(meta.get('formattedDistance', '0')),
            
            # Ubicaciones clave para el análisis en Morelia y tu app YAGA
            'origen': meta.get('pickupAddress', 'Ubicación no disponible'),
            'destino': meta.get('dropOffAddress', 'Ubicación no disponible'),
            'url_mapa': meta.get('mapUrl', ''),
            
            'estado': self.raw.get('status', '')
        }

In [40]:
import json
# Cargar el archivo que generaste con el scraper
with open("viajes_marzo_capturados.json", "r", encoding="utf-8") as f:
    datos_crudos = json.load(f)

# Inicializar y generar reporte
comparador = UberTripManager(datos_crudos)
comparador.generar_comparativa_json()


📊 RESUMEN DESDE ARCHIVO JSON: MARZO 2025
🗓️ SEMANA 9: Del 24/02/2025 al 02/03/2025
 🚗 VIAJES:           84
 ⏱️ TIEMPO (VIAJES):   18h 10m
 💵 MONTO BRUTO:      $4,458.39
-------------------------------------------------------
🗓️ SEMANA 10: Del 03/03/2025 al 09/03/2025
 🚗 VIAJES:           82
 ⏱️ TIEMPO (VIAJES):   18h 26m
 💵 MONTO BRUTO:      $4,595.21
-------------------------------------------------------
🗓️ SEMANA 11: Del 10/03/2025 al 16/03/2025
 🚗 VIAJES:           110
 ⏱️ TIEMPO (VIAJES):   21h 19m
 💵 MONTO BRUTO:      $5,249.90
-------------------------------------------------------
🗓️ SEMANA 12: Del 17/03/2025 al 23/03/2025
 🚗 VIAJES:           71
 ⏱️ TIEMPO (VIAJES):   13h 13m
 💵 MONTO BRUTO:      $3,432.38
-------------------------------------------------------
🗓️ SEMANA 13: Del 24/03/2025 al 30/03/2025
 🚗 VIAJES:           112
 ⏱️ TIEMPO (VIAJES):   21h 16m
 💵 MONTO BRUTO:      $4,826.11
-------------------------------------------------------
🗓️ SEMANA 14: Del 31/03/2025 al 

In [41]:
class UberAuditorPrecision:
    """
    Clase para auditar la precisión entre los datos extraídos de PDF 
    y los capturados vía JSON/Playwright.
    """
    def __init__(self, pdf_data, json_data):
        self.pdf = pdf_data
        self.json = json_data

    def calcular_exactitud(self, valor_real, valor_estimado):
        if valor_real == 0: return 0
        # Fórmula de exactitud: 1 - |(Real - Estimado) / Real|
        return (1 - abs(valor_real - valor_estimado) / valor_real) * 100

    def generar_reporte_auditoria(self):
        # Cálculos de precisión
        p_ingresos = self.calcular_exactitud(self.pdf['monto'], self.json['monto'])
        p_viajes = self.calcular_exactitud(self.pdf['viajes'], self.json['viajes'])
        
        delta_monto = self.json['monto'] - self.pdf['monto']
        delta_viajes = self.json['viajes'] - self.pdf['viajes']

        print("="*60)
        print(f"⚖️ AUDITORÍA DE DATOS: {self.pdf['mes'].upper()} {self.pdf['anio']}")
        print("="*60)
        
        print(f"💰 PRECISIÓN FINANCIERA: {p_ingresos:.2f}%")
        print(f"   Diferencia: ${delta_monto:,.2f} MXN")
        print(f"   Nota: {'Captura excedente (Posible propina/cancelación)' if delta_monto > 0 else 'Fuga de datos'}")
        
        print(f"\n🚗 PRECISIÓN DE ACTIVIDAD: {p_viajes:.2f}%")
        print(f"   Diferencia: {delta_viajes} eventos (viajes/cancelaciones)")
        
        print("-" * 60)
        exactitud_global = (p_ingresos + p_viajes) / 2
        print(f"🏆 SCORE DE CONFIANZA GLOBAL: {exactitud_global:.2f}%")
        print("="*60)

# =========================================================================
# EJECUCIÓN CON TUS DATOS DE MARZO
# =========================================================================
datos_pdf = {
    'mes': 'Marzo', 'anio': 2025,
    'monto': 25840.97, 'viajes': 492
}

datos_json = {
    'monto': 25891.23, 'viajes': 531
}

auditor = UberAuditorPrecision(datos_pdf, datos_json)
auditor.generar_reporte_auditoria()

⚖️ AUDITORÍA DE DATOS: MARZO 2025
💰 PRECISIÓN FINANCIERA: 99.81%
   Diferencia: $50.26 MXN
   Nota: Captura excedente (Posible propina/cancelación)

🚗 PRECISIÓN DE ACTIVIDAD: 92.07%
   Diferencia: 39 eventos (viajes/cancelaciones)
------------------------------------------------------------
🏆 SCORE DE CONFIANZA GLOBAL: 95.94%


In [33]:
import pandas as pd
import json
import re
from datetime import datetime, timezone, timedelta

class UberDatasetGenerator:
    def __init__(self, json_path="viajes_marzo_capturados.json"):
        self.json_path = json_path
        self.semanas_marzo = [9, 10, 11, 12, 13, 14]
        self.anio_objetivo = 2025

    @staticmethod
    def _parse_money(text: str) -> float:
        if not text: return 0.0
        cleaned = re.sub(r'[^\d\.,]', '', text)
        if ',' in cleaned and '.' in cleaned:
            cleaned = cleaned.replace('.', '').replace(',', '.') if cleaned.rfind(',') > cleaned.rfind('.') else cleaned.replace(',', '')
        elif ',' in cleaned:
            cleaned = cleaned.replace(',', '.') if len(cleaned) - cleaned.rfind(',') == 3 else cleaned.replace(',', '')
        try: return float(cleaned)
        except: return 0.0

    @staticmethod
    def _parse_duration(text: str) -> float:
        if not text: return 0.0
        m = re.search(r'(\d+)\s*min', text)
        s = re.search(r'(\d+)\s*seg', text)
        total = int(m.group(1)) if m else 0
        total += (int(s.group(1)) / 60) if s else 0
        return round(total, 2)

    def generar_dataset(self):
        with open(self.json_path, "r", encoding="utf-8") as f:
            raw_data = json.load(f)

        viajes_limpios = []
        for item in raw_data:
            ts_utc = datetime.fromtimestamp(int(item.get('recognizedAt', 0)), tz=timezone.utc)
            # Ajuste Morelia (UTC-6) + Corte Uber (4 AM) = -10 horas
            ts_uber = ts_utc - timedelta(hours=10)
            
            meta = item.get('tripMetaData') or {}
            
            # Solo incluimos el rango de marzo 2025
            if ts_uber.year == self.anio_objetivo and ts_uber.isocalendar().week in self.semanas_marzo:
                monto = self._parse_money(item.get('formattedTotal', ''))
                duracion = self._parse_duration(meta.get('formattedDuration', ''))
                
                viajes_limpios.append({
                    'trip_id': item.get('uuid'),
                    'fecha_local': ts_uber.strftime('%Y-%m-%d %H:%M:%S'),
                    'dia_semana': ts_uber.strftime('%A'),
                    'semana_iso': ts_uber.isocalendar().week,
                    'monto_bruto': monto,
                    'duracion_minutos': duracion,
                    'distancia_txt': meta.get('formattedDistance', '0 km'),
                    'estado': item.get('status'),
                    # Métrica de eficiencia para tu análisis
                    'mxn_por_minuto': round(monto / duracion, 2) if duracion > 0 else 0
                })

        df = pd.DataFrame(viajes_limpios).drop_duplicates(subset=['trip_id'])
        df.to_csv("dataset_uber_marzo_2025.csv", index=False)
        print(f"✅ Dataset de Marzo generado: {len(df)} registros guardados.")
        return df

# Ejecución
gen = UberDatasetGenerator()
df_marzo = gen.generar_dataset()

✅ Dataset de Marzo generado: 531 registros guardados.


In [45]:
import json

# 1. Carga del archivo JSON capturado por el scraper
# Asegúrate de que el nombre del archivo coincida con el que guardaste
with open("viajes_marzo_capturados.json", "r", encoding="utf-8") as f:
    datos_crudos = json.load(f)

# 2. Definición de las semanas fiscales de Marzo 2025 (alineado con tus PDFs)
semanas_marzo = [9, 10, 11, 12, 13, 14]

# 3. Procesamiento y Filtrado
viajes_procesados = []
for raw in datos_crudos:
    # Instanciamos la clase y obtenemos el diccionario limpio
    actividad = TripActivity(raw).data
    
    # Filtro estricto: Solo viajes de Marzo 2025 con ingreso positivo
    if actividad['year'] == 2025 and actividad['semana'] in semanas_marzo:
        if actividad['monto'] > 0:
            viajes_procesados.append(actividad)

# 4. Creación del DataFrame de Pandas
df_marzo_2025 = pd.DataFrame(viajes_procesados)

# Eliminamos duplicados por ID de viaje (seguridad de datos)
if not df_marzo_2025.empty:
    df_marzo_2025 = df_marzo_2025.drop_duplicates(subset=['trip_id'])

print(f"✅ Dataset listo: Se han consolidado {len(df_marzo_2025)} registros para Marzo.")
df_marzo_2025.head()

✅ Dataset listo: Se han consolidado 497 registros para Marzo.


,trip_id,year,semana,fecha_local,monto,duracion_min,distancia_km,origen,destino,url_mapa,estado
0,463d9fbe-a914-466b-a08b-dcdba8018cad,2025,9,2025-03-02 18:16:50,83.50,22.00,8.78,"20 de Noviembre, Morelia, 58000 Michoacán, MX","Morelia, 58359 Morelia, Michoacán de Ocampo, MX",https://static-maps.uber.com/map?width=360&hei...,COMPLETED
1,19671a48-4b83-4494-bc45-110a50e0ed6e,2025,9,2025-03-02 18:08:21,33.42,7.33,4.09,"Maro, Morelia, 58254 Michoacán, MX","Av. del Estudiante, Morelia, 58228 Michoacán, MX",https://static-maps.uber.com/map?width=360&hei...,COMPLETED
2,a9dc1c48-6a0f-42ef-8b1a-4018331b3f29,2025,9,2025-03-02 17:37:58,47.40,13.67,5.87,"Calle Totorames, 58227 Morelia, Michoacán de O...","Perif. Paseo de la República, Morelia, 58270 M...",https://static-maps.uber.com/map?width=360&hei...,COMPLETED
3,ee5d2a01-7d62-4939-b10c-85f2472b659b,2025,9,2025-03-02 17:20:14,70.40,9.98,4.01,"Calle Homero, 58257 Morelia, Michoacán de Ocam...","Calle Fray Sebastian de Aparicio, 58227 Moreli...",https://static-maps.uber.com/map?width=360&hei...,COMPLETED
4,62d7df70-e6fb-4176-ba49-cd386fbb3020,2025,9,2025-03-02 16:58:57,72.37,13.17,10.58,"Perif Paseo de la República, 58148 Morelia, MI...","Av Acueducto, Morelia, 58270 Michoacán, MX",https://static-maps.uber.com/map?width=360&hei...,COMPLETED


In [46]:
# 1. Enriquecimiento final de columnas
df_marzo_2025['fecha_local'] = pd.to_datetime(df_marzo_2025['fecha_local'])
df_marzo_2025['hora'] = df_marzo_2025['fecha_local'].dt.hour
df_marzo_2025['dia_semana'] = df_marzo_2025['fecha_local'].dt.day_name()

# 2. Métricas de eficiencia para YAGA
df_marzo_2025['mxn_por_minuto'] = (df_marzo_2025['monto'] / df_marzo_2025['duracion_min']).round(2)
df_marzo_2025['monto_neto_est'] = (df_marzo_2025['monto'] * 0.75).round(2) # Estimación post-comisión Uber

# 3. Clasificación de Turnos (Contexto Morelia)
def asignar_turno(h):
    if 5 <= h < 12: return 'Mañana'
    if 12 <= h < 18: return 'Tarde'
    if 18 <= h < 24: return 'Noche'
    return 'Madrugada'

df_marzo_2025['turno'] = df_marzo_2025['hora'].apply(asignar_turno)

# Mostrar el Dataset Completo (Estructura y Muestra)
print(f"✅ DATASET TOTAL: {len(df_marzo_2025)} Registros")
print("-" * 30)
display(df_marzo_2025.head(10)) # Muestra de los primeros 10 viajes

✅ DATASET TOTAL: 497 Registros
------------------------------


,trip_id,year,semana,fecha_local,monto,duracion_min,distancia_km,origen,destino,url_mapa,estado,hora,dia_semana,mxn_por_minuto,monto_neto_est,turno
0,463d9fbe-a914-466b-a08b-dcdba8018cad,2025,9,2025-03-02 18:16:50,83.50,22.00,8.78,"20 de Noviembre, Morelia, 58000 Michoacán, MX","Morelia, 58359 Morelia, Michoacán de Ocampo, MX",https://static-maps.uber.com/map?width=360&hei...,COMPLETED,18,Sunday,3.80,62.62,Noche
1,19671a48-4b83-4494-bc45-110a50e0ed6e,2025,9,2025-03-02 18:08:21,33.42,7.33,4.09,"Maro, Morelia, 58254 Michoacán, MX","Av. del Estudiante, Morelia, 58228 Michoacán, MX",https://static-maps.uber.com/map?width=360&hei...,COMPLETED,18,Sunday,4.56,25.06,Noche
2,a9dc1c48-6a0f-42ef-8b1a-4018331b3f29,2025,9,2025-03-02 17:37:58,47.40,13.67,5.87,"Calle Totorames, 58227 Morelia, Michoacán de O...","Perif. Paseo de la República, Morelia, 58270 M...",https://static-maps.uber.com/map?width=360&hei...,COMPLETED,17,Sunday,3.47,35.55,Tarde
3,ee5d2a01-7d62-4939-b10c-85f2472b659b,2025,9,2025-03-02 17:20:14,70.40,9.98,4.01,"Calle Homero, 58257 Morelia, Michoacán de Ocam...","Calle Fray Sebastian de Aparicio, 58227 Moreli...",https://static-maps.uber.com/map?width=360&hei...,COMPLETED,17,Sunday,7.05,52.80,Tarde
4,62d7df70-e6fb-4176-ba49-cd386fbb3020,2025,9,2025-03-02 16:58:57,72.37,13.17,10.58,"Perif Paseo de la República, 58148 Morelia, MI...","Av Acueducto, Morelia, 58270 Michoacán, MX",https://static-maps.uber.com/map?width=360&hei...,COMPLETED,16,Sunday,5.50,54.28,Tarde
5,4b70093d-fbc1-453c-84c3-617c58d2c16f,2025,9,2025-03-02 16:44:59,49.36,15.62,5.01,"Calle General Martín Castrejón, 58040 Morelia,...","Ruperto Castañeda, Morelia, 58146 Michoacán, MX",https://static-maps.uber.com/map?width=360&hei...,COMPLETED,16,Sunday,3.16,37.02,Tarde
6,21c48a7c-0cd2-4f84-b42c-ec427444b4e7,2025,9,2025-03-02 16:40:44,6.52,0.00,0.00,"Calle Virrey de Mendoza, 58020 Morelia, Michoa...","Calle Virrey de Mendoza, 58020 Morelia, Michoa...",https://static-maps.uber.com/map?width=360&hei...,CANCELED,16,Sunday,inf,4.89,Tarde
7,443ac1b9-8713-4ac6-ae37-6218a072f02c,2025,9,2025-03-02 16:19:50,36.42,8.75,2.78,"Calz La Huerta, Morelia, 58080 Michoacán, MX","Av Solidaridad, Morelia, 58030 Michoacán, MX",https://static-maps.uber.com/map?width=360&hei...,COMPLETED,16,Sunday,4.16,27.32,Tarde
8,0e81dac3-f44c-404f-aa6d-446f513fa052,2025,9,2025-03-02 16:06:06,124.72,13.80,5.79,"Calle Jilguero, 58341 Tenencia Morelos, Michoa...","Calz La Huerta, Morelia, 58080 Michoacán, MX",https://static-maps.uber.com/map?width=360&hei...,COMPLETED,16,Sunday,9.04,93.54,Tarde
9,78a2993d-821c-4eda-b69f-d3dde426bdb6,2025,9,2025-03-02 00:45:21,54.36,14.00,6.19,"Carretera Huajúmbaro - San José, 58891 Cuitzil...","Jazmin, 58893 Tarímbaro, Michoacán de Ocampo, MX",https://static-maps.uber.com/map?width=360&hei...,COMPLETED,0,Sunday,3.88,40.77,Madrugada


In [47]:
# Muestra los primeros 5 registros con todas las columnas procesadas
df_marzo_2025.head(5)

,trip_id,year,semana,fecha_local,monto,duracion_min,distancia_km,origen,destino,url_mapa,estado,hora,dia_semana,mxn_por_minuto,monto_neto_est,turno
0,463d9fbe-a914-466b-a08b-dcdba8018cad,2025,9,2025-03-02 18:16:50,83.50,22.00,8.78,"20 de Noviembre, Morelia, 58000 Michoacán, MX","Morelia, 58359 Morelia, Michoacán de Ocampo, MX",https://static-maps.uber.com/map?width=360&hei...,COMPLETED,18,Sunday,3.80,62.62,Noche
1,19671a48-4b83-4494-bc45-110a50e0ed6e,2025,9,2025-03-02 18:08:21,33.42,7.33,4.09,"Maro, Morelia, 58254 Michoacán, MX","Av. del Estudiante, Morelia, 58228 Michoacán, MX",https://static-maps.uber.com/map?width=360&hei...,COMPLETED,18,Sunday,4.56,25.06,Noche
2,a9dc1c48-6a0f-42ef-8b1a-4018331b3f29,2025,9,2025-03-02 17:37:58,47.40,13.67,5.87,"Calle Totorames, 58227 Morelia, Michoacán de O...","Perif. Paseo de la República, Morelia, 58270 M...",https://static-maps.uber.com/map?width=360&hei...,COMPLETED,17,Sunday,3.47,35.55,Tarde
3,ee5d2a01-7d62-4939-b10c-85f2472b659b,2025,9,2025-03-02 17:20:14,70.40,9.98,4.01,"Calle Homero, 58257 Morelia, Michoacán de Ocam...","Calle Fray Sebastian de Aparicio, 58227 Moreli...",https://static-maps.uber.com/map?width=360&hei...,COMPLETED,17,Sunday,7.05,52.80,Tarde
4,62d7df70-e6fb-4176-ba49-cd386fbb3020,2025,9,2025-03-02 16:58:57,72.37,13.17,10.58,"Perif Paseo de la República, 58148 Morelia, MI...","Av Acueducto, Morelia, 58270 Michoacán, MX",https://static-maps.uber.com/map?width=360&hei...,COMPLETED,16,Sunday,5.50,54.28,Tarde


In [48]:
# Muestra la URL completa del mapa para el quinto registro (índice 4)
print(df_marzo_2025.iloc[4]['url_mapa'])

https://static-maps.uber.com/map?width=360&height=100&marker=lat%3A19.71924%24lng%3A-101.22612%24icon%3Ahttps%3A%2F%2Fd1a3f4spazzrp4.cloudfront.net%2Fmaps%2Fhelix%2Fcar-pickup-pin.png%24anchorX%3A0.5%24anchorY%3A1.0&marker=lat%3A19.69517%24lng%3A-101.15636%24icon%3Ahttps%3A%2F%2Fd1a3f4spazzrp4.cloudfront.net%2Fmaps%2Fhelix%2Fcar-dropoff-pin.png%24anchorX%3A0.5%24anchorY%3A1.0&polyline=color%3A0xFF2DBAE4%24width%3A4%24enc%3AeljwBfvyhRD_BeB%3FqDtFmRkJwPwKiC%7BEa%40iFvRm%7EC%60%60%40yjAY%7BFcEmKPmE%60%7B%40ynAzUoRvg%40oq%40%7CAiAzZfBc%40%7CWt%40mO&source=769a605b_1c9b_3da4_a486_ccc264a18174&signature=-Um9TIfFYa5M2uOjj3UbG2MgFXU=


In [49]:
import urllib.parse

def crear_url_google_maps(origen, destino):
    """
    Construye una URL de navegación de Google Maps usando las direcciones.
    """
    if not origen or not destino or "Ubicación no disponible" in origen:
        return None
    
    # Codificamos las direcciones para que sean seguras en una URL
    base_url = "https://www.google.com/maps/dir/?api=1"
    query_origen = urllib.parse.quote(origen)
    query_destino = urllib.parse.quote(destino)
    
    return f"{base_url}&origin={query_origen}&destination={query_destino}"

# 1. Aplicamos la función para crear la nueva columna
df_marzo_2025['enlace_google_maps'] = df_marzo_2025.apply(
    lambda x: crear_url_google_maps(x['origen'], x['destino']), axis=1
)

# 2. Bloque de TESTING: Mostramos los primeros 5 enlaces
print("🧪 PRUEBA DE ENLACES INTERACTIVOS (PRIMEROS 5):")
print("-" * 50)
for i, url in enumerate(df_marzo_2025['enlace_google_maps'].head(5), 1):
    print(f"Viaje {i}: {url}\n")

🧪 PRUEBA DE ENLACES INTERACTIVOS (PRIMEROS 5):
--------------------------------------------------
Viaje 1: https://www.google.com/maps/dir/?api=1&origin=20%20de%20Noviembre%2C%20Morelia%2C%2058000%20Michoac%C3%A1n%2C%20MX&destination=Morelia%2C%2058359%20Morelia%2C%20Michoac%C3%A1n%20de%20Ocampo%2C%20MX

Viaje 2: https://www.google.com/maps/dir/?api=1&origin=Maro%2C%20Morelia%2C%2058254%20Michoac%C3%A1n%2C%20MX&destination=Av.%20del%20Estudiante%2C%20Morelia%2C%2058228%20Michoac%C3%A1n%2C%20MX

Viaje 3: https://www.google.com/maps/dir/?api=1&origin=Calle%20Totorames%2C%2058227%20Morelia%2C%20Michoac%C3%A1n%20de%20Ocampo%2C%20MX&destination=Perif.%20Paseo%20de%20la%20Rep%C3%BAblica%2C%20Morelia%2C%2058270%20Michoac%C3%A1n%2C%20MX

Viaje 4: https://www.google.com/maps/dir/?api=1&origin=Calle%20Homero%2C%2058257%20Morelia%2C%20Michoac%C3%A1n%20de%20Ocampo%2C%20MX&destination=Calle%20Fray%20Sebastian%20de%20Aparicio%2C%2058227%20Morelia%2C%20Michoac%C3%A1n%20de%20Ocampo%2C%20MX

Viaje 5: h

In [50]:
import urllib.parse

def generar_url_interactiva(origen, destino):
    """Genera un enlace de Google Maps Directions para navegación real."""
    if not origen or not destino or "Ubicación no disponible" in origen:
        return None
    
    # Codificación de caracteres especiales (espacios, comas, etc.)
    base_url = "https://www.google.com/maps/dir/?api=1"
    query_origen = urllib.parse.quote(origen)
    query_destino = urllib.parse.quote(destino)
    
    return f"{base_url}&origin={query_origen}&destination={query_destino}"

# 1. Creamos la columna en el DataFrame
df_marzo_2025['enlace_google_maps'] = df_marzo_2025.apply(
    lambda x: generar_url_interactiva(x['origen'], x['destino']), axis=1
)

# 2. Reordenamos las columnas para que el enlace sea fácil de encontrar
columnas = ['fecha_local', 'monto', 'origen', 'destino', 'enlace_google_maps', 'trip_id']
df_vista = df_marzo_2025[columnas]

print(f"✅ Enlaces generados para {len(df_marzo_2025)} registros.")
display(df_vista.head())

✅ Enlaces generados para 497 registros.


,fecha_local,monto,origen,destino,enlace_google_maps,trip_id
0,2025-03-02 18:16:50,83.50,"20 de Noviembre, Morelia, 58000 Michoacán, MX","Morelia, 58359 Morelia, Michoacán de Ocampo, MX",https://www.google.com/maps/dir/?api=1&origin=...,463d9fbe-a914-466b-a08b-dcdba8018cad
1,2025-03-02 18:08:21,33.42,"Maro, Morelia, 58254 Michoacán, MX","Av. del Estudiante, Morelia, 58228 Michoacán, MX",https://www.google.com/maps/dir/?api=1&origin=...,19671a48-4b83-4494-bc45-110a50e0ed6e
2,2025-03-02 17:37:58,47.40,"Calle Totorames, 58227 Morelia, Michoacán de O...","Perif. Paseo de la República, Morelia, 58270 M...",https://www.google.com/maps/dir/?api=1&origin=...,a9dc1c48-6a0f-42ef-8b1a-4018331b3f29
3,2025-03-02 17:20:14,70.40,"Calle Homero, 58257 Morelia, Michoacán de Ocam...","Calle Fray Sebastian de Aparicio, 58227 Moreli...",https://www.google.com/maps/dir/?api=1&origin=...,ee5d2a01-7d62-4939-b10c-85f2472b659b
4,2025-03-02 16:58:57,72.37,"Perif Paseo de la República, 58148 Morelia, MI...","Av Acueducto, Morelia, 58270 Michoacán, MX",https://www.google.com/maps/dir/?api=1&origin=...,62d7df70-e6fb-4176-ba49-cd386fbb3020


In [52]:
df_marzo_2025.to_csv("dataset_uber_marzo_maps.csv", index=False)

In [53]:
import json
import re
import urllib.parse
import pandas as pd
from datetime import datetime, timezone, timedelta

class UberTripProcessor:
    def __init__(self, anio_objetivo=2025, semanas_objetivo=None):
        self.anio = anio_objetivo
        self.semanas = semanas_objetivo or [9, 10, 11, 12, 13, 14]

    @staticmethod
    def _parse_money(text: str) -> float:
        if not text: return 0.0
        cleaned = re.sub(r'[^\d\.,]', '', text)
        if ',' in cleaned and '.' in cleaned:
            cleaned = cleaned.replace('.', '').replace(',', '.') if cleaned.rfind(',') > cleaned.rfind('.') else cleaned.replace(',', '')
        elif ',' in cleaned:
            cleaned = cleaned.replace(',', '.') if len(cleaned) - cleaned.rfind(',') == 3 else cleaned.replace(',', '')
        try: return float(cleaned)
        except: return 0.0

    @staticmethod
    def _parse_duration(text: str) -> float:
        if not text: return 0.0
        m = re.search(r'(\d+)\s*min', text)
        s = re.search(r'(\d+)\s*seg', text)
        total = int(m.group(1)) if m else 0
        total += (int(s.group(1)) / 60) if s else 0
        return round(total, 2)

    @staticmethod
    def _generar_url_maps(origen, destino):
        if not origen or not destino or "Ubicación no disponible" in origen:
            return None
        base_url = "https://www.google.com/maps/dir/?api=1"
        return f"{base_url}&origin={urllib.parse.quote(origen)}&destination={urllib.parse.quote(destino)}"

    def procesar_archivo(self, json_path):
        with open(json_path, "r", encoding="utf-8") as f:
            datos_crudos = json.load(f)

        viajes_limpios = []
        for item in datos_crudos:
            ts_utc = datetime.fromtimestamp(int(item.get('recognizedAt', 0)), tz=timezone.utc)
            # Ajuste de 10 horas para alinear con el corte de las 4 AM en Morelia
            ts_uber = ts_utc - timedelta(hours=10)
            
            if ts_uber.year == self.anio and ts_uber.isocalendar().week in self.semanas:
                meta = item.get('tripMetaData') or {}
                monto = self._parse_money(item.get('formattedTotal', ''))
                duracion = self._parse_duration(meta.get('formattedDuration', ''))
                origen = meta.get('pickupAddress', 'Ubicación no disponible')
                destino = meta.get('dropOffAddress', 'Ubicación no disponible')

                viajes_limpios.append({
                    'trip_id': item.get('uuid'),
                    'fecha_local': ts_uber.strftime('%Y-%m-%d %H:%M:%S'),
                    'semana_iso': ts_uber.isocalendar().week,
                    'monto_bruto': monto,
                    'monto_neto_est': round(monto * 0.75, 2),
                    'duracion_min': duracion,
                    'distancia_km': self._parse_money(meta.get('formattedDistance', '0')),
                    'origen': origen,
                    'destino': destino,
                    'enlace_google_maps': self._generar_url_maps(origen, destino),
                    'estado': item.get('status')
                })

        df = pd.DataFrame(viajes_limpios).drop_duplicates(subset=['trip_id'])
        return df

# --- EJECUCIÓN PARA MARZO ---
procesador = UberTripProcessor(anio_objetivo=2025, semanas_objetivo=[9, 10, 11, 12, 13, 14])
df_marzo_final = procesador.procesar_archivo("viajes_marzo_capturados.json")

# Guardar el dataset listo para auditoría y para YAGA
df_marzo_final.to_csv("dataset_uber_marzo_2025_completo.csv", index=False)
print(f"✅ Dataset de Marzo generado con {len(df_marzo_final)} registros y enlaces de Maps.")

✅ Dataset de Marzo generado con 531 registros y enlaces de Maps.


In [54]:
# Cargar el dataset recién creado
df = pd.read_csv("dataset_uber_marzo_2025_completo.csv")

print("--- ESTRUCTURA Y CALIDAD ---")
print(f"Total de viajes: {len(df)}")
print(f"Columnas disponibles: {df.columns.tolist()}")
print("\nValores faltantes por columna:")
print(df.isnull().sum())

--- ESTRUCTURA Y CALIDAD ---
Total de viajes: 531
Columnas disponibles: ['trip_id', 'fecha_local', 'semana_iso', 'monto_bruto', 'monto_neto_est', 'duracion_min', 'distancia_km', 'origen', 'destino', 'enlace_google_maps', 'estado']

Valores faltantes por columna:
trip_id               0
fecha_local           0
semana_iso            0
monto_bruto           0
monto_neto_est        0
duracion_min          0
distancia_km          0
origen                0
destino               0
enlace_google_maps    3
estado                3
dtype: int64


In [55]:
print("\n--- DESGLOSE POR ESTADO ---")
# Esto te mostrará cuántos son 'COMPLETED' vs 'CANCELED' con pago
print(df['estado'].value_counts())


--- DESGLOSE POR ESTADO ---
estado
COMPLETED          488
CANCELED            34
DRIVER_CANCELED      6
Name: count, dtype: int64


In [56]:
print("📍 TOP 5 PUNTOS DE RECOLECCIÓN (PICKUP):")
print(df_marzo_final['origen'].value_counts().head(5))

print("\n🏁 TOP 5 DESTINOS (DROPOFF):")
print(df_marzo_final['destino'].value_counts().head(5))

📍 TOP 5 PUNTOS DE RECOLECCIÓN (PICKUP):
origen
Perif Paseo de la República, 58148 Morelia, MIC, MX                      5
Paseo de la República, 58270 Morelia, Michoacán de Ocampo, MX            4
Calle Francisco J. Múgica, 58040 Morelia, Michoacán de Ocampo, MX        4
Av. Madero Poniente #5, 58000 Morelia, Michoacán de Ocampo, MX           4
Juan Jose de Lejarza, Morelia, 58000 Morelia, Michoacán de Ocampo, MX    3
Name: count, dtype: int64

🏁 TOP 5 DESTINOS (DROPOFF):
destino
Perif. Paseo de la República, Morelia, 58148 Michoacán, MX           13
Av Francisco I. Madero Pte, Morelia, 58000 Michoacán, MX              5
Avenida Francisco I. Madero Oriente, Morelia, 58000 Michoacán, MX     4
Morelia, 58000 Michoacán, MX                                          4
Av Lázaro Cárdenas, Morelia, 58260 Michoacán, MX                      3
Name: count, dtype: int64


In [59]:
!pip install geopy folium

In [61]:
import folium
from folium.plugins import HeatMap, MarkerCluster
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd
import time

# 1. Preparar el geocodificador
geolocator = Nominatim(user_agent="yaga_app_analyzer")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# NOTA: Geocodificar 531 direcciones puede tardar unos 10-15 min. 
# Vamos a probar con una muestra de los primeros 50 para el test inicial.
df_test = df_marzo_final.head(50).copy()

print("🌍 Geocodificando puntos de origen (Morelia)...")
df_test['location'] = df_test['origen'].apply(geocode)
df_test['lat'] = df_test['location'].apply(lambda loc: loc.latitude if loc else None)
df_test['lon'] = df_test['location'].apply(lambda loc: loc.longitude if loc else None)

# Filtramos los que no se pudieron geocodificar
df_geo = df_test.dropna(subset=['lat', 'lon'])

🌍 Geocodificando puntos de origen (Morelia)...
